# Imports and Setup

In [10]:
import os
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, recall_score, precision_score, f1_score, balanced_accuracy_score, classification_report
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

In [6]:
# Setting up the root directory
root_dir = r"C:\Users\Source\OneDrive - IESEG\Desktop\MBD\ML\Kaggle_Competition"

os.chdir(root_dir)

os.getcwd()

'C:\\Users\\Source\\OneDrive - IESEG\\Desktop\\MBD\\ML\\Kaggle_Competition'

# II - Load Data

### Loading the cleaned data

In [7]:
train = pd.read_csv(os.path.join(root_dir, 'data', 'processed', 'train_preprocessed.csv'))
train_labels = pd.read_csv(os.path.join(root_dir, 'data', 'processed', 'y_train.csv'))
test = pd.read_csv(os.path.join(root_dir, 'data', 'processed', 'test_preprocessed.csv'))

In [8]:
# Verifying the imports
display(train.head(3))
display(train_labels.head(3))
display(test.head(3))

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var14822_freq,Var14893_freq,Var14904_freq,Var14910_freq,Var14913_freq,Var14923_freq,Var14965_freq,Var14970_freq,Var14990_freq,Var14993_freq
0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,...,0.000167,0.000833,0.000033,0.959833,0.000233,0.5867,0.071467,0.079633,0.082333,0.000833
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000067,0.001433,0.007033,0.959833,0.000600,0.5867,0.005167,0.347167,0.082333,0.001433
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000033,0.000167,0.002667,0.959833,0.000100,0.5867,0.002233,0.347167,0.028400,0.000167


,Target_appetency
0,0
1,0
2,0


,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var14822_freq,Var14893_freq,Var14904_freq,Var14910_freq,Var14913_freq,Var14923_freq,Var14965_freq,Var14970_freq,Var14990_freq,Var14993_freq
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000667,0.000600,0.001300,0.959833,0.000700,0.128267,0.089000,0.057333,0.162700,0.000600
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.001333,0.001767,0.001100,0.959833,0.001367,0.128267,0.009433,0.023033,0.022400,0.001767
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000400,0.011567,0.959833,0.000100,0.128267,0.006467,0.014467,0.027333,0.000400


# III - KNN Classifier

In [15]:
X_train = train.copy()
y_train = train_labels['Target_appetency'].values

# We define our pipeline
# We use imblearn pipeline instead of sklear pipeline as we will use SMOTE during fit and not transform, which is not supported by sklearn
pipeline = Pipeline([
    ('scaler', StandardScaler()), # We scale inside out CV to avoid leakage (and not in preprocessing !)
    ('pca', PCA(random_state=42)), # We 
    ('smote', SMOTE(random_state=42)),
    ('knn', KNeighborsClassifier())
])

# Defining the tuning parameters
param_grid = {
    'pca__n_components' : [30,50,100],
    'knn__n_neighbors' : [3, 5, 11, 21, 51],
    'knn__metric' : ['euclidean', 'manhattan', 'minkowski'],
    'knn__weights' : ['uniform', 'distance']
}

# K-fold with stratified (to preserve the minority class ratio)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid seach with scoring using the AUC
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=1,
    verbose=2,
    refit=True  # refit best model on full train at the end
)

# fitting
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 90 candidates, totalling 450 fits
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  20.9s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  20.1s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  17.0s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  17.5s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=30; total time=  16.6s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=50; total time=  19.1s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=50; total time=  21.7s
[CV] END knn__metric=euclidean, knn__n_neighbors=3, knn__weights=uniform, pca__n_components=50; total time=  20.5s
[CV] END knn__metr

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'knn__metric': ['euclidean', 'manhattan', ...], 'knn__n_neighbors': [3, 5, ...], 'knn__weights': ['uniform', 'distance'], 'pca__n_components': [30, 50, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the m

In [ ]:
# Printing the results 
print(f'Best AUC (CV) : {grid_search.best_score_}')
print(f'Best params: {grid_search.best_params_}')

Best AUC (CV) : 0.6455880284461158
Best params : {'knn__metric': 'manhattan', 'knn__n_neighbors': 51, 'knn__weights': 'distance', 'pca__n_components': 100}


In [17]:
# Predicting on test set

y_pred_proba = grid_search.best_estimator_.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'ID': test.index,
    'Target_appetency': y_pred_proba
})
submission.to_csv('output/submission_knn.csv', index=False)